#### cNMF runs using the raw counts matrix (UMIs), without any TPM-like normalization.

In [ ]:
# Setup 
%matplotlib inline

import os
import pandas as pd
import numpy as np
from scipy.io import mmread
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import Image
import scanpy as sc
import seaborn as sns
import pickle
import glob

In [ ]:
# Base directories (local runs and UGER runs).
basedir = "/home/EOCRC_atlas"

# Location of anndata object with all YOCRC epithelial data. ALL RAW DATA, NEEDS SUBSETTING
file_h5ad_all = f"{basedir}/data/all_samples_raw_withTier2Annotation.h5ad"

# Directory name settings.
date = "DATE"
name_project = "EOCRC"

# Output directories 
figuresdir = "%s/results/%s_%s_cNMF/" % (basedir, date, name_project)
outputdir_NMF = "%s/cNMF" % figuresdir
srcdir_NMF = "/home/EOCRC_atlas/src/cNMF/"

# Save gene expression program variables created by cnmf in a pickle file.
filename_pi = f"{figuresdir}/{date}_{name_project}_{age}_GEP_objects.P"
filename_clustermap = f"{figuresdir}/{date}_{name_project}_{age}_clustermap_objects.P"

# Make directories.
if not os.path.exists(figuresdir):
    os.mkdir(figuresdir)
if not os.path.exists(outputdir_NMF):
    os.mkdir(outputdir_NMF)

# Specify which age window to group NMF programs together. Script runs NMF on all samples, but grouping is done on young or old.
age_min = 0
age_max = 50

# Settings for cnmf prepare and factorize scripts.
numiter = 100 # Number of NMF replicates. Set this to a larger value ~200 for real data. We set this to a relatively low value here for illustration at a faster speed
numworkers = 4 # Number of parallel factorization jobs to run. Set this to a value reflective of the number of cores on your computer.
numhvgenes = 2000 ## Number of over-dispersed genes to use for running the cnmf factorizations
seed = 14 ## Specify a seed pseudorandom number generation for reproducibility

# Specify the number of factors (Ks) to use as a space separated list in this case "5 6 7 8 9 10" by cnmf prepare.
K = ' '.join([str(i) for i in range(4, 10)])

# Get list of protein coding genes in GRCh38. I am not interested in lncRNA and antisense transcripts, 
# and so I am filtering these out from the expression matrices, using
file_codinggenes = "/home/EOCRC_atlas/src/cNMF/GRCh38-3.0.0_proteincoding.txt"
coding_genes = [line.rstrip() for line in open(file_codinggenes, "r")]

# Downsample? 
max_sample_size = "Inf"  

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=80)
sc.settings.figdir = figuresdir

In [ ]:
# Read anndata object with all the data, which must have raw counts data. Then subset to epithelial cells.
adata_all = sc.read_h5ad(file_h5ad_all)

# Filter to only non mixed cell types 
run_celltypes = ['Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium']

adata_all = adata_all[adata_all.obs['Annotation_Tier2'].isin(run_celltypes)] # subset to only non mixed cell types 
adata_all = adata_all[adata_all.obs['Annotation_Tier1']=="Epithelial"] # subset to only epithelial 

# Generate sampleids based on the samples in the anndata object.
sampleids = list(adata_all.obs.FRID.unique())

# Check for sampleids that have too few epithelial cells 
cell_counts = adata_all.obs['FRID'].value_counts()
cell_counts

In [ ]:
# Remove any specific samples that did not pass quality control.
#sampleids_remove = ['COLFR0301_T1', 'COLFR0366_T1']  # removed due to too few epithelial cells
sampleids_remove = cell_counts[cell_counts <100].index.tolist()
if len(sampleids_remove):
    sampleids = [i for i in sampleids if i not in sampleids_remove]

# Get all samples within a specific age window.
sampleids_age = np.unique(adata_all.obs['FRID'][adata_all.obs['Cohort']==age])
len(sampleids_age)

### ONLY RUN ONCE - Make an anndata object for each individual patient sample.
This code chunk divides the full anndata object with all the epithelium data into individual anndata object and verifies we are storing raw count data, which is required by cNMF. Furthermore, lncRNAs and antisense transcripts are removed from the counts matrix, as we do not want to interpret these as part of gene programs.

In [ ]:
# Iterate over each sample, remove non protein-coding genes from counts matrix, subset on malignant or non-malignant cells,
# calculate variable genes (optional), subsample cells (optional), write updated anndata counts matrix.
for sampleid in sampleids:
    # h5ad files storing count matrices and filtered count matrices for each individual sample.
    file_h5ad_subset = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    
    # Subset to cells from individual sample.
    cells = adata_all.obs[adata_all.obs['FRID'] == sampleid].index.tolist()
    adata = adata_all[cells, :]
    
    # Keep protein coding genes and remove antisense genes and lncRNA genes from count matrices.
    subset_genes = list(set(coding_genes) & set(adata.var.index))
    adata = adata[:, subset_genes]
    print(adata)

    # Write updated anndata object to file.
    adata.write(file_h5ad_subset)

### ONLY RUN ONCE - Prepare each individual patient anndata object as input to cnmf prepare.
lncRNAs and antisense transcripts were previously removed from the counts matrix. Variables genes can be calculated here with scanpy, or use existing ones, or let cnmf prepare calculate them. cNMF requires raw count matries. Any sample with fewer than 100 cells after these filtering steps is removed from sampleids and samplesheet since cNMF will probably not be informative for these samples. Two samples were removed: ['COLFR0301_T1', 'COLFR0366_T1']

In [ ]:
# List of sampleids to remove (based on too few cells being captured).
sampleids_remove = []

# Iterate over each sample, remove non protein-coding genes from counts matrix, subset on malignant or non-malignant cells,
# calculate variable genes (optional), subsample cells (optional), write updated anndata counts matrix.
for sampleid in sampleids:
    # h5ad files storing count matrices and filtered count matrices.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)

    # Read in the anndata object, which must have raw counts data.
    adata = sc.read_h5ad(file_h5ad)
    print(adata.n_obs)
    
    # Highly variable genes were already identified when generating the Seurat object.
    # Only use highly variable genes that are coding genes.
    # hvg = adata.var[adata.var['Selected'] == 1].index.tolist()
    # hvg = list(set(hvg) & set(coding_genes))
    # print(len(hvg))

    # Highly variable genes may also be identified by scanpy.
    # Only use highly variable genes that are coding genes.
    print(np.sum(adata.X.toarray(), axis = 1))  # check current normalization scheme
    adata_copy = adata.copy()  # keep a copy of the raw count data
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=5, min_disp=0.5, flavor='seurat')
    # sc.pl.highly_variable_genes(adata)
    hvg = adata.var[adata.var['highly_variable']].index.tolist()
    hvg = list(set(hvg) & set(coding_genes))
    adata = adata_copy  # return to using the raw count data, which is what is expected by cNMF
    print(len(hvg))
    
    # Write highly variable genes to file.
    file_hvg = "%s/%s_hvg.txt" % (figuresdir, sampleid)
    with open(file_hvg, "w") as f:
        f.write("\n".join(hvg))
    
    # Subset sample to a specified number of cells, in order to speed up calculations.
    if max_sample_size != "Inf":
        sample_size = min(max_sample_size, adata.n_obs)  # if number of cells is less than subsample size, use all cells
        sc.pp.subsample(adata, n_obs=sample_size, random_state=0)

    # Remove genes not expressed in any of the final, filtered cells, as this can interfere with cNMF according to Dylan Kotliar.
    sc.pp.filter_genes(adata, min_cells=3)

    # Write updated anndata object to file.
    adata.write(file_h5ad)

    # If there are fewer than 100 cells, remove this sample from further analysis.
    if adata.n_obs < 100:
        sampleids_remove.append(sampleid)

# Remove any samples that have fewer than 100 cells from further analysis.
if len(sampleids_remove):
    sampleids = [i for i in sampleids if i not in sampleids_remove]

In [ ]:
# Iterate over each sample.
from cnmf import cNMF
import os 
import cnmf.cnmf
import warnings
cnmf.cnmf.warnings = warnings # had to add this - what are the warning issues? 


for sampleid in sampleids:
    print(sampleid)

    # code to pick up when a run was interrupted - skip samples that have already been run 
    sample_output_dir = os.path.join(figuresdir, sampleid)
    skip_sample = False
    if os.path.exists(sample_output_dir):
        for root, dirs, files in os.walk(sample_output_dir):
            if any(f.endswith('merged.df.npz') for f in files):
                print(f"Skipping {sampleid} because a file ending in 'merged.df.npz' already exists in {root}")
                skip_sample = True
                break
    if skip_sample:
        continue

    # h5ad files storing count matrices.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    file_hvg = "%s/%s_hvg.txt" % (figuresdir, sampleid)
    
    # Commands for running cNMF from python - compare to Orr Ashenberg's UGER commands 
    cnmf_obj = cNMF(output_dir=figuresdir, name=sampleid)
    cnmf_obj.prepare(counts_fn=file_h5ad, components=np.arange(4,16), n_iter=100, seed=14, num_highvar_genes=numhvgenes)

    numworkers = 4
    factorize_cmd = f'nohup parallel python /home/cporter/ENTER/envs/scanpy/lib/python3.12/site-packages/cnmf/cnmf.py factorize --output-dir {figuresdir} --name {sampleid} --worker-index {{}} ::: 0 1 2 3'
    print('Factorize command to simultaneously run factorization over %d cores using GNU parallel:\n%s' % (numworkers, factorize_cmd))
    os.system(factorize_cmd)
    
    #cnmf_obj.factorize(worker_i=0, total_workers=1)
    cnmf_obj.combine()
    cnmf_obj.k_selection_plot(close_fig=False)

In [ ]:
# Iterate over each sample, run PCA, and make scree plot.
for sampleid in sampleids:
    run_name = sampleid
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)

    # Variable gene selection.
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=5, min_disp=0.5, flavor='seurat')
    sc.pp.scale(adata, max_value=10)

    # Do PCA on highly variable genes and make scree (elbow) plot.
    # adata.var['highly_variable'] = [True if i == 1 else False for i in adata.var['Selected']]
    sc.tl.pca(adata, use_highly_variable = True)
    sc.settings.figdir =  "%s/%s" % (outputdir_NMF, run_name)
    sc.pl.pca_variance_ratio(adata, save = "_%s.png" % run_name)

In [ ]:
from cnmf import cNMF
import os 
selected_K_list = [10] * len(sampleids)  # same K for all samples
density_threshold = 2.0
density_threshold_str = ('%.1f' % density_threshold).replace('.', '_')

# Iterate over each sample and show consensus.
for i, sampleid in enumerate(sampleids):
    run_name = sampleid  # NMF results will be saved to [outputdir_NMF]/[run_name] 
    selected_K = selected_K_list[i]
    
    cnmf_obj = cNMF(output_dir=figuresdir, name=sampleid)

    ## This is the command you would run from the command line to obtain the consensus estimate with no filtering
    ## and to save a diagnostic plot as a PDF
    cnmf_obj.consensus(k=selected_K, density_threshold=density_threshold)

    print(sampleid)
    display(Image(filename = "%s/%s/%s.clustering.k_%d.dt_%s.png" % (figuresdir, run_name, run_name, selected_K, density_threshold_str),
          width=800, height=800))

In [ ]:
from cnmf import cNMF
import os 
import cnmf.cnmf
import warnings
cnmf.cnmf.warnings = warnings # had to add this - what are the warning issues? 

density_threshold = .2
selected_K_list = [10] * len(sampleids)  # same K for all samples

density_threshold_str = ('%.1f' % density_threshold).replace('.', '_')

# Iterate over each sample and show consensus.
for i, sampleid in enumerate(sampleids):
    run_name = sampleid  # NMF results will be saved to [outputdir_NMF]/[run_name] 
    selected_K = selected_K_list[i]
    
    cnmf_obj = cNMF(output_dir=figuresdir, name=sampleid)

    ## This is the command you would run from the command line to obtain the consensus estimate with filtering
    ## and to save a diagnostic plot as a PDF
    cnmf_obj.consensus(k=selected_K, density_threshold=density_threshold)

    print(sampleid)
    display(Image(filename = "%s/%s/%s.clustering.k_%d.dt_%s.png" % (figuresdir, run_name, run_name, selected_K, density_threshold_str),
          width=800, height=800))

#### A number of output files have been created by the consensus step for cNMF.

In [ ]:
# Iterate over each sample and show output files.
for sampleid in sampleids:
    run_name = sampleid
    print(os.listdir("%s/%s/" % (figuresdir, run_name)))

### Load the raw factor usage files and gene spectra files that were output by cNMF.
We are most interested in the usage.consensus.txt and gene_spectra_score.txt files for the density threshold of 0.2:

Below, we load those in as text files, append them to a processed Scanpy AnnData object so we can make plots of how the usages overlap with UMAP dimensionality reudctions of the data

cNMF does not normalize the usages to sum to 1 for each cell so we do that as a subsequent step in the second cell below


In [ ]:
# Iterate over each sample and save usage and gene spectra scores to anndata object.
for i, sampleid in enumerate(sampleids):
    run_name = sampleid
    selected_K = selected_K_list[i]
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)

    # Remove any Usage or GEP columns that were previously added to these dataframes.
    adata.obs = adata.obs.drop(columns = adata.obs.columns[adata.obs.columns.str.startswith("Usage")])
    adata.var = adata.var.drop(columns = adata.var.columns[adata.var.columns.str.startswith("GEP")])

    #################################
    # Load the raw factor usage file
    #################################
    # Read in raw usage file and normalize factor usage values within each cell to sum to 1.
    usage = pd.read_csv("%s/%s/%s.usages.k_%d.dt_%s.consensus.txt" % (figuresdir, run_name, run_name, selected_K, density_threshold_str),sep='\t', index_col=0)

    usage.columns = ['Usage_%s' % i for i in usage.columns]
    usage.head()

    # Normalize usages to sum to 1 within each cell.
    usage_norm = usage.div(usage.sum(axis=1), axis=0)
    usage_norm.head()

    # Add normalized usages to anndata observations dataframe.
    adata.obs = pd.merge(left=adata.obs, right=usage_norm, how='left', left_index=True, right_index=True)
    adata.obs.head()

    #################################
    # Load in the gene_scores and identify the genes that are most associated with each program.
    #################################
    #  Load the Z-scored GEPs which reflect how enriched a gene is in each GEP relative to all of the other genes.
    gene_scores = pd.read_csv("%s/%s/%s.gene_spectra_score.k_%d.dt_%s.txt" % (figuresdir, run_name, run_name, selected_K, density_threshold_str),sep='\t', index_col=0).T
    gene_scores.columns = ['GEP_zscore_%s' % i for i in gene_scores.columns]
    gene_scores.head()

    # Load the TPM GEPs
    gene_tpms = pd.read_csv("%s/%s/%s.gene_spectra_tpm.k_%d.dt_%s.txt" % (figuresdir, run_name, run_name, selected_K, density_threshold_str),sep='\t', index_col=0).T
    gene_tpms.columns = ['GEP_tpm_%s' % i for i in gene_tpms.columns]

    # Add TPM GEPs and linear regression B coeffs Z-score GEPs to anndata variables dataframe.
    adata.var = pd.merge(left=adata.var, right=gene_scores, how='left', left_index=True, right_index=True)
    adata.var = pd.merge(left=adata.var, right=gene_tpms, how='left', left_index=True, right_index=True)
    adata.var.head()

    # Save an updated h5ad file with cNMF results.
    adata.write(file_h5ad)

### Plot usage scores on UMAPs for individual patient samples and show top genes for each program in each patient.

In [ ]:
# Iterate over each sample and show usage and gene spectra scores.
for sampleid in sampleids:
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    
    adata = sc.read(file_h5ad)
    
    # Read in usage scores and gene spectra scores.
    usage_norm = adata.obs.loc[:, adata.obs.columns.str.startswith("Usage_")]
    gene_scores = adata.var.loc[:, adata.var.columns.str.startswith("GEP_zscore")]  # Z-scored GEPs
    
    # Read in highly variables genes that were used for cNMF.
    hvgs = open(f"{figuresdir}/{sampleid}/{sampleid}.overdispersed_genes.txt").read().split('\n')
    
    ## Set log-normalized data to the raw attribute of the AnnData object to make it easy to plot expression levels of individual genes.
    ## This does not log normalize the actual AnnData data matrix
    sc.pp.normalize_total(adata, target_sum=1e4)
    adata.raw = sc.pp.log1p(adata.copy(), copy=True)  # Set log-normalized data to the raw attribute of the AnnData object

    # Dimensionality reduction and UMAP.
    sc.pp.log1p(adata)
    adata = adata[:,hvgs] # Subset out only the high-variance genes
    sc.pp.scale(adata)  # Mean and variance normalize the genes
    sc.pp.pca(adata) # Run PCA
    # sc.pl.pca_variance_ratio(adata, log=True)  # Make a scree plot to determine number of PCs to use for UMAP
    sc.pp.neighbors(adata, n_neighbors=50, n_pcs=40)  # Construct the nearest neighbor graph for UMAP
    sc.tl.umap(adata)  # Run UMAP
        
    # Show topgenes for each gene expression program using Z-scored GEPs.
    ngenes = 100
    topgenes = []
    for gep in gene_scores.columns:
        topgenes.append(list(gene_scores.sort_values(by=gep, ascending=False).index[:ngenes]))
    topgenes = pd.DataFrame(topgenes, index=gene_scores.columns).T
    topgenes.to_csv(f"{figuresdir}/{sampleid}_genespectra_topgenes.csv", index=False)
    print(sampleid)
    display(topgenes)
    
    # Show marker genes and usage scores on UMAP.
    with plt.rc_context():  # Use this to set figure params like size and dpi
        try:
            sc.pl.umap(adata, color=['EPCAM', 'LGR5','PCNA', 'PTPRC'], ncols=2, use_raw=True, show=False)
            plt.savefig(f"{figuresdir}/{sampleid}_umap_markers.png", bbox_inches="tight")
            plt.show()
        except KeyError:
            print(f"{sampleid} adata is missing some marker genes (after hvg) when making feature plot.")
        sc.pl.umap(adata, color=usage_norm.columns, ncols=3, vmin=0, vmax=1, show=False)
        plt.savefig(f"{figuresdir}/{sampleid}_umap_usage.png", bbox_inches="tight")
        plt.show()
        sc.pl.umap(adata, color="Annotation_Tier2", vmin=0, vmax=1, show=False)
        plt.savefig(f"{figuresdir}/{sampleid}_umap_Annotation_Tier2.png", bbox_inches="tight")
        plt.show()

    # Save UMAP for individual sample into anndata object.
    pca = adata.obsm['X_pca']
    umap = adata.obsm['X_umap']
    adata = sc.read(file_h5ad)  # return to original anndata object and add dimensionality reductions
    adata.obsm['X_pca'] = pca
    adata.obsm['X_umap'] = umap
    adata.write(file_h5ad)

## Subset to MSS Left - pick up here for testing different k and clustering resolutions 

In [ ]:
adata_all = adata_all[(adata_all.obs['MSI_v2']=="MSS: STABLE") & (adata_all.obs['Sidedness']=="Left")]

### Make sample ID lists by age

In [ ]:
sampleids_young= np.unique(adata_all.obs['FRID'][adata_all.obs['Cohort']=="UnderFifty"])
print(len(sampleids_young))
sampleids_remove = cell_counts[cell_counts <100].index.tolist()
if len(sampleids_remove):
    sampleids_young = [i for i in sampleids_young if i not in sampleids_remove]
    print(len(sampleids_young))
          
sampleids_old= np.unique(adata_all.obs['FRID'][adata_all.obs['Cohort']=="FiftyPlus"])
print(len(sampleids_old))
if len(sampleids_remove):
    sampleids_old = [i for i in sampleids_old if i not in sampleids_remove]
    print(len(sampleids_old))

## Young samples

### Collect gene TPM and spectra scores (gene weight in factor) that were calculated for each sample.
Collect all GEPs calculated from all individual samples, and place them into a single dataframe. Collect both the TMP and zscore beta coefficients. This enables clustering the GEP programs across all individuals.

In [ ]:
# Dataframe where row is gene and column is GEP from a sample. Includes GEPs identified in all samples.
# We collect both the TPM and beta coefficients (zscore).
GEP_tpm_samples = pd.DataFrame()
GEP_zscore_samples = pd.DataFrame()

for sampleid in sampleids_young:
    run_name = sampleid
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)

    # Get gene expression program columns, add unique sampleid, and merge programs across all samples for spectra z scores.
    gene_scores = adata.var
    gene_scores = gene_scores.loc[:, gene_scores.columns.str.startswith("GEP_zscore")]
    gene_scores.columns = ['%s_%s' % (sampleid, i) for i in gene_scores.columns]
    if GEP_zscore_samples.columns.empty:
        GEP_zscore_samples = gene_scores
    else:
        GEP_zscore_samples = pd.merge(left=GEP_zscore_samples, right=gene_scores, how='outer', left_index=True, right_index=True)  # merge on gene names
        # GEP_zscore_samples = pd.merge(left=GEP_zscore_samples, right=gene_scores, how='inner', left_index=True, right_index=True)  # merge on gene names

    # Get gene expression program columns, add unique sampleid, and merge programs across all samples for TPMs.
    gene_scores = adata.var
    gene_scores = gene_scores.loc[:, gene_scores.columns.str.startswith("GEP_tpm")]  
    gene_scores.columns = ['%s_%s' % (sampleid, i) for i in gene_scores.columns]
    if GEP_tpm_samples.columns.empty:
        GEP_tpm_samples = gene_scores
    else:
        GEP_tpm_samples = pd.merge(left=GEP_tpm_samples, right=gene_scores, how='outer', left_index=True, right_index=True)  # merge on gene names
        # GEP_tpm_samples = pd.merge(left=GEP_tpm_samples, right=gene_scores, how='inner', left_index=True, right_index=True)  # merge on gene names


In [ ]:
# For all samples, obtain the top genes for each GEP in sorted order and combine them into a single dataframe
topgenes_GEP_samples = []
ngenes = 50
gene_scores = GEP_zscore_samples
# gene_scores = GEP_tpm_samples
for gep in gene_scores.columns:
    topgenes_GEP_samples.append(list(gene_scores.sort_values(by=gep, ascending=False).index[:ngenes]))
topgenes_GEP_samples = pd.DataFrame(topgenes_GEP_samples, index=gene_scores.columns).T
topgenes_GEP_samples

### Cluster GEP programs: first score all cells using the top scoring genes in all GEPs.
Each set of programs from each individual is now defined by the top genes in the program. Score all cells from all individuals using those top genes in each program.

In [ ]:
# Dataframe where row is GEP and column is cell from a sample. Value is the expression of the GEP program in the cell. 
# This includes GEPs identified across all samples and cells from all samples.
cellscores_GEP_samples = pd.DataFrame()
age = "UnderFifty"
ncells = 0
# Iterate over each sample. 
for sampleid in sampleids_young:
    # Read in data, normalize, and log-transform the data.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    adata.raw = sc.pp.log1p(adata.copy(), copy=True)
    ncells += adata.n_obs
    
    # Score cells with top genes from each GEP identified across all samples.
    for gep in topgenes_GEP_samples.columns:
        gene_list = topgenes_GEP_samples[gep].tolist()
        try:
            sc.tl.score_genes(adata, gene_list, ctrl_size=50, n_bins=25, score_name=gep, use_raw=True)
        except ValueError:  #  ValueError in score_genes() if no top genes expressed in this anndata object
            print(f"{gep} genes are not expressed in {sampleid}.")
            adata.obs[gep] = 0

    # Add GEP cell scores for this sample to cellscores_GEP_samples dataframe.
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("GEP")]
    cell_scores = adata.obs[gep_columns].T
    if cellscores_GEP_samples.columns.empty:
        cellscores_GEP_samples = cell_scores
    else:
        cellscores_GEP_samples = pd.concat([cellscores_GEP_samples, cell_scores], axis=1)
    
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("%s_GEP" % sampleid)]
    sc.pl.umap(adata, color=gep_columns, use_raw=True, ncols=4)
    #sc.pl.umap(adata, color = ["leiden_labels", "SingleR_cluster_labels"]) 

# Write cellscores_GEP_samples to file.
file_name = f"{figuresdir}/cellscores_{age}_GEP_samples.P"
cellscores_GEP_samples.to_pickle(file_name)


### Cluster GEP programs: second cluster cells by their GEPs cell scores, and identify groups of GEPs that cluster together from multiple patient samples

In [ ]:
tmp = adata.obs.columns[adata.obs.columns.str.contains("_GEP")]
tmp2 = [col.split('_')[0] for col in tmp]
np.unique(tmp2)

In [ ]:
# Hierarchical/agglomerative clustering using scipy.cluster.hierarchy.linkage
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster

age = "UnderFifty"

file_name = f"{figuresdir}/cellscores_{age}_GEP_samples.P"
data = pd.read_pickle(file_name)
X = np.array(data)

# Cluster rows (GEPs) by similarity in GEP scoring across all individual cells.
# Pairwise distance metric between programs is 1-correlation. Row linkage (GEPs).
dist = distance.pdist(X, metric = 'correlation')
row_linkage = hierarchy.linkage(dist, method='ward', metric='correlation')

# Cluster columns (samples) by average expression of each GEP in cells from sample (# GEPs across samples x # samples).
# This is faster than calculating across hundreds of thousands of individual cells.
data_plot = data.copy().drop(labels=data.columns, axis=1)  # make empty dataframe with no columns, only GEP names as rows
cols_sampleid = [i.split("-")[0] for i in data.columns.to_list()]  # sampleid for each cell 
for id in list(set(cols_sampleid)):  # for each individual sample, calculate the mean of each GEP across all cells in sample
    cols = data.columns[[i for i in range(len(cols_sampleid)) if cols_sampleid[i] == id]]  # columns of all cells in sample
    data_plot[id] = data[cols].mean(axis=1)
col_linkage = hierarchy.linkage(distance.pdist(np.array(data_plot).T, metric = 'correlation'), method='ward', metric='correlation')

# Cut the dendrogram linkage tree to assign similar GEPs to clusters.
# The t parameter sets where the tree is cut, and therefore the number of clusters.
clusters_GEP = fcluster(row_linkage, t=4, criterion='distance')  # 2.8 for young, 3.8 for old

df_clusters_GEP = pd.DataFrame({'sample': data.index, 'cluster': clusters_GEP})
df_clusters_GEP.sort_values(by = "cluster", inplace = True)

# Define row colors based on which cluster GEP was assigned to.
network_pal = sns.husl_palette(len(df_clusters_GEP['cluster'].unique()), s=.45)  # map each cluster to a color
lut = dict(zip(df_clusters_GEP['cluster'].unique(), network_pal))
row_colors = df_clusters_GEP['cluster'].map(lut)
row_colors.index = df_clusters_GEP['sample']

# Define row colors based on which patient sample the GEP came from.
# samples = pd.Series([i.split("_GEP")[0] for i in data.index], index = data.index)  # e.g. get COLFR0552_T1
# network_pal = sns.husl_palette(len(samples.unique()), s=.45)  # map each sample to a color
# lut = dict(zip(samples.unique(), network_pal))
# row_colors = samples.map(lut)

# Define column colors based on patient sampleid.
network_pal = network_pal = sns.color_palette("viridis", len(data_plot.columns))  # map each sampleid to a color
col_colors = pd.Series(network_pal, index=data_plot.columns, name = "sample")

# Make clustermap using linkage matrices calculated above. consensus GEP program x patient sampleid
g = sns.clustermap(data_plot, row_linkage=row_linkage, col_linkage=col_linkage, row_colors=row_colors, col_colors=col_colors,
                   xticklabels = True, yticklabels = True, cmap = sns.color_palette("Reds"), figsize=(20,40), 
                   z_score=0, vmin=-3, vmax=3)
sns.set(font_scale=0.5)
g.ax_col_dendrogram.set_visible(False)
#g.savefig('%s/cluster_cellscores_%s_GEPs.png' % (figuresdir, age), bbox_inches='tight')
#g.savefig('%s/cluster_cellscores_%s_GEPs_11-7-25.pdf' % (figuresdir, age), bbox_inches='tight')

plt.show()
# Row names (GEPs across all patient samples) listed in the same order as the clustermap.
data.index[g.dendrogram_row.reordered_ind]

# Look only at first 10 clusters
# df_clusters_GEP = df_clusters_GEP.loc[df_clusters_GEP['cluster'].isin(range(10)), :]
# df_clusters_GEP.loc[df_clusters_GEP['cluster'] == 1, :]

# Save cluster map variables in a pickle.
#object_pi = {"clustermap": g, "cellscores_GEP_samples": data}
#file_pi = open(filename_clustermap, "wb") 
#pickle.dump(object_pi, file_pi)
#file_pi.close()

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import squareform

# Convert your compact pdist vector into a square distance matrix
distance_matrix = squareform(dist)

# Set the GEP string names as the index so .loc works flawlessly
df_clusters_GEP = df_clusters_GEP.set_index('sample')

# Extract your assigned cluster labels matching the matrix order exactly
cluster_labels = df_clusters_GEP.loc[data.index, 'cluster'].values

# Put the index back to a column just in case your downstream code expects it that way
df_clusters_GEP = df_clusters_GEP.reset_index()

# Compute the overall mean score for your paper
mean_sil = silhouette_score(distance_matrix, cluster_labels, metric='precomputed')
print(f"--- Silhouette Analysis for {age} ---")
print(f"Overall Average Silhouette Score: {mean_sil:.4f}\n")

# Compute individual scores for every single raw GEP program
sample_sil_values = silhouette_samples(distance_matrix, cluster_labels, metric='precomputed')

# Build a summary DataFrame for easy sorting and plotting
df_sil = pd.DataFrame({
    'GEP': data.index,
    'Cluster': cluster_labels,
    'Silhouette_Score': sample_sil_values
})

# Sort the programs primarily by Cluster, and secondarily by Silhouette Score descending
df_sil = df_sil.sort_values(by=['Cluster', 'Silhouette_Score'], ascending=[True, False]).reset_index(drop=True)

# Print average score per cluster to identify any under-clustered structural weaknesses
for i in sorted(df_sil['Cluster'].unique()):
    cluster_avg = df_sil[df_sil['Cluster'] == i]['Silhouette_Score'].mean()
    print(f"  * Mean Silhouette for Meta-Program Cluster {i}: {cluster_avg:.4f}")

### Find top genes for consensus GEPs that cluster across multiple patients and visualize them
Plot consensus GEPs on UMAP of each patient sample.

In [ ]:
age = "UnderFifty"

gene_scores = GEP_zscore_samples
# gene_scores = GEP_tpm_samples  # column names do not match GEPs in df_clusters_GEP currently
# gene_scores.columns = GEP_zscore_samples.columns

# Identify mitochondrial genes, and drop them.
# genes_MT = gene_scores.index[gene_scores.index.str.contains("^MT-")].tolist()
# gene_scores.drop(genes_MT, axis=0, inplace=True)

# Group GEPs by assigned cluster from agglomerative clustering above, take average of each gene coefficient across GEPs, 
# sort genes by their average, keep top genes.
topgenes_GEP_consensus = []  # store consensus GEP programs[]
ngenes = 50
for name, group in df_clusters_GEP.groupby('cluster'):
    # Store GEPs, which were clustered together, in a dataframe.
    geps = group['sample']
    df_GEP = gene_scores[geps]
    
    # Get mean of each gene NMF score across all GEPs in the cluster, and sort genes by their mean.
    df_GEP.loc[:, 'mean'] = df_GEP.mean(axis = 1)
    df_GEP.sort_values(by='mean', ascending=False, inplace=True)
    topgenes_GEP_consensus.append(list(df_GEP.index[:ngenes]))

# Give names to consensus GEPs.
row_names = ['GEP_consensus_%s' % (i+1) for i in range(df_clusters_GEP['cluster'].nunique())]
topgenes_GEP_consensus = pd.DataFrame(topgenes_GEP_consensus, index=row_names).T
#topgenes_GEP_consensus.to_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus.csv", index=True)
topgenes_GEP_consensus.to_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus_SIX_META_CLUSTERS.csv", index=True)

# Place all top genes in a single list.
list_topgenes = []
for i in topgenes_GEP_consensus.columns:
    list_topgenes.extend(topgenes_GEP_consensus[i].tolist())

# Iterate over each sample and score each cell with the topgenes_GEP_consensus. 
for sampleid in sampleids_young:
    # Read in data, normalize, and log-transform the data.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    adata.raw = sc.pp.log1p(adata.copy(), copy=True)
    
    # Score cells with each GEP identified across all samples.
    for gep in topgenes_GEP_consensus.columns:
        gene_list = topgenes_GEP_consensus[gep].tolist()
        try:
            sc.tl.score_genes(adata, gene_list, ctrl_size=50, n_bins=25, score_name=gep, use_raw=True)
        except ValueError:  #  ValueError in score_genes() if no top genes expressed in this anndata object
            print(f"{gep} genes are not expressed in {sampleid}.")
            adata.obs[gep] = 0

    # UMAP and heatmap visualizations of genes.
    print(sampleid)
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("GEP_consensus")]
    sc.pl.umap(adata, color=gep_columns, use_raw=True, ncols=4, size = 12) 
    # sc.pl.umap(adata, color = ["leiden_labels", "SingleR_cluster_labels"], size = 12) 
    # sc.pl.heatmap(adata, var_names = list_topgenes, groupby="SingleR_cluster_labels")
    
# Save gene expression program variables in a pickle.
object_pi = {"topgenes_GEP_samples": topgenes_GEP_samples, "cellscores_GEP_samples": data, 
             "GEP_zscore_samples": GEP_zscore_samples, "topgenes_GEP_consensus": topgenes_GEP_consensus, "clustermap": g}
# file_pi = open(filename_pi, "wb") 
# pickle.dump(object_pi, file_pi)
# file_pi.close()

# Code to load data from pickled object.
# file_pi = open(filename_pi, "rb") 
# object_pi = pickle.load(file_pi)
# file_pi.close()
# GEP_zscore_samples = object_pi["GEP_zscore_samples"]

# Fifty plus

### Collect gene TPM and spectra scores (gene weight in factor) that were calculated for each sample.
Collect all GEPs calculated from all individual samples, and place them into a single dataframe. Collect both the TMP and zscore beta coefficients. This enables clustering the GEP programs across all individuals.

In [ ]:
# Dataframe where row is gene and column is GEP from a sample. Includes GEPs identified in all samples.
# We collect both the TPM and beta coefficients (zscore).
GEP_tpm_samples = pd.DataFrame()
GEP_zscore_samples = pd.DataFrame()

for sampleid in sampleids_old:
    run_name = sampleid
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)

    # Get gene expression program columns, add unique sampleid, and merge programs across all samples for spectra z scores.
    gene_scores = adata.var
    gene_scores = gene_scores.loc[:, gene_scores.columns.str.startswith("GEP_zscore")]
    gene_scores.columns = ['%s_%s' % (sampleid, i) for i in gene_scores.columns]
    if GEP_zscore_samples.columns.empty:
        GEP_zscore_samples = gene_scores
    else:
        GEP_zscore_samples = pd.merge(left=GEP_zscore_samples, right=gene_scores, how='outer', left_index=True, right_index=True)  # merge on gene names
        # GEP_zscore_samples = pd.merge(left=GEP_zscore_samples, right=gene_scores, how='inner', left_index=True, right_index=True)  # merge on gene names

    # Get gene expression program columns, add unique sampleid, and merge programs across all samples for TPMs.
    gene_scores = adata.var
    gene_scores = gene_scores.loc[:, gene_scores.columns.str.startswith("GEP_tpm")]  
    gene_scores.columns = ['%s_%s' % (sampleid, i) for i in gene_scores.columns]
    if GEP_tpm_samples.columns.empty:
        GEP_tpm_samples = gene_scores
    else:
        GEP_tpm_samples = pd.merge(left=GEP_tpm_samples, right=gene_scores, how='outer', left_index=True, right_index=True)  # merge on gene names
        # GEP_tpm_samples = pd.merge(left=GEP_tpm_samples, right=gene_scores, how='inner', left_index=True, right_index=True)  # merge on gene names


In [ ]:
# For all samples, obtain the top genes for each GEP in sorted order and combine them into a single dataframe
topgenes_GEP_samples = []
ngenes = 50
gene_scores = GEP_zscore_samples
# gene_scores = GEP_tpm_samples
for gep in gene_scores.columns:
    topgenes_GEP_samples.append(list(gene_scores.sort_values(by=gep, ascending=False).index[:ngenes]))
topgenes_GEP_samples = pd.DataFrame(topgenes_GEP_samples, index=gene_scores.columns).T
topgenes_GEP_samples

### Cluster GEP programs: first score all cells using the top scoring genes in all GEPs.
Each set of programs from each individual is now defined by the top genes in the program. Score all cells from all individuals using those top genes in each program.

In [ ]:
# Dataframe where row is GEP and column is cell from a sample. Value is the expression of the GEP program in the cell. 
# This includes GEPs identified across all samples and cells from all samples.
cellscores_GEP_samples = pd.DataFrame()
age = "FiftyPlus"
ncells = 0
# Iterate over each sample. 
for sampleid in sampleids_old:
    # Read in data, normalize, and log-transform the data.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    adata.raw = sc.pp.log1p(adata.copy(), copy=True)
    ncells += adata.n_obs
    
    # Score cells with top genes from each GEP identified across all samples.
    for gep in topgenes_GEP_samples.columns:
        gene_list = topgenes_GEP_samples[gep].tolist()
        try:
            sc.tl.score_genes(adata, gene_list, ctrl_size=50, n_bins=25, score_name=gep, use_raw=True)
        except ValueError:  #  ValueError in score_genes() if no top genes expressed in this anndata object
            print(f"{gep} genes are not expressed in {sampleid}.")
            adata.obs[gep] = 0

    # Add GEP cell scores for this sample to cellscores_GEP_samples dataframe.
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("GEP")]
    cell_scores = adata.obs[gep_columns].T
    if cellscores_GEP_samples.columns.empty:
        cellscores_GEP_samples = cell_scores
    else:
        cellscores_GEP_samples = pd.concat([cellscores_GEP_samples, cell_scores], axis=1)
    
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("%s_GEP" % sampleid)]
    sc.pl.umap(adata, color=gep_columns, use_raw=True, ncols=4)
    #sc.pl.umap(adata, color = ["leiden_labels", "SingleR_cluster_labels"]) 

# Write cellscores_GEP_samples to file.
file_name = f"{figuresdir}/cellscores_{age}_GEP_samples.P"
cellscores_GEP_samples.to_pickle(file_name)


### Cluster GEP programs: second cluster cells by their GEPs cell scores, and identify groups of GEPs that cluster together from multiple patient samples

In [ ]:
# Hierarchical/agglomerative clustering using scipy.cluster.hierarchy.linkage
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster

age = "FiftyPlus"

file_name = f"{figuresdir}/cellscores_{age}_GEP_samples.P"
data = pd.read_pickle(file_name)
X = np.array(data)

# Cluster rows (GEPs) by similarity in GEP scoring across all individual cells.
# Pairwise distance metric between programs is 1-correlation. Row linkage (GEPs).
dist = distance.pdist(X, metric = 'correlation')
row_linkage = hierarchy.linkage(dist, method='ward', metric='correlation')

# Cluster columns (samples) by average expression of each GEP in cells from sample (# GEPs across samples x # samples).
# This is faster than calculating across hundreds of thousands of individual cells.
data_plot = data.copy().drop(labels=data.columns, axis=1)  # make empty dataframe with no columns, only GEP names as rows
cols_sampleid = [i.split("-")[0] for i in data.columns.to_list()]  # sampleid for each cell 
for id in list(set(cols_sampleid)):  # for each individual sample, calculate the mean of each GEP across all cells in sample
    cols = data.columns[[i for i in range(len(cols_sampleid)) if cols_sampleid[i] == id]]  # columns of all cells in sample
    data_plot[id] = data[cols].mean(axis=1)
col_linkage = hierarchy.linkage(distance.pdist(np.array(data_plot).T, metric = 'correlation'), method='ward', metric='correlation')

# Cut the dendrogram linkage tree to assign similar GEPs to clusters.
# The t parameter sets where the tree is cut, and therefore the number of clusters.
clusters_GEP = fcluster(row_linkage, t=4, criterion='distance')  # 2.8 for young, 3.8 for old
clusters_GEP = fcluster(row_linkage, t=3, criterion='distance')  # 2.8 for young, 3.8 for old

df_clusters_GEP = pd.DataFrame({'sample': data.index, 'cluster': clusters_GEP})
df_clusters_GEP.sort_values(by = "cluster", inplace = True)

# Define row colors based on which cluster GEP was assigned to.
network_pal = sns.husl_palette(len(df_clusters_GEP['cluster'].unique()), s=.45)  # map each cluster to a color
lut = dict(zip(df_clusters_GEP['cluster'].unique(), network_pal))
row_colors = df_clusters_GEP['cluster'].map(lut)
row_colors.index = df_clusters_GEP['sample']

# Define row colors based on which patient sample the GEP came from.
# samples = pd.Series([i.split("_GEP")[0] for i in data.index], index = data.index)  # e.g. get COLFR0552_T1
# network_pal = sns.husl_palette(len(samples.unique()), s=.45)  # map each sample to a color
# lut = dict(zip(samples.unique(), network_pal))
# row_colors = samples.map(lut)

# Define column colors based on patient sampleid.
network_pal = network_pal = sns.color_palette("viridis", len(data_plot.columns))  # map each sampleid to a color
col_colors = pd.Series(network_pal, index=data_plot.columns, name = "sample")

# Make clustermap using linkage matrices calculated above. consensus GEP program x patient sampleid
g = sns.clustermap(data_plot, row_linkage=row_linkage, col_linkage=col_linkage, row_colors=row_colors, col_colors=col_colors,
                   xticklabels = True, yticklabels = True, cmap = sns.color_palette("Reds"), figsize=(20,40), 
                   z_score=0, vmin=-3, vmax=3)
sns.set(font_scale=0.5)
g.ax_col_dendrogram.set_visible(False)
#g.savefig('%s/cluster_cellscores_%s_GEPs.png' % (figuresdir, age), bbox_inches='tight')
#g.savefig('%s/cluster_cellscores_%s_GEPs_11-7-25.pdf' % (figuresdir, age), bbox_inches='tight')

plt.show()
# Row names (GEPs across all patient samples) listed in the same order as the clustermap.
data.index[g.dendrogram_row.reordered_ind]

# Look only at first 10 clusters
# df_clusters_GEP = df_clusters_GEP.loc[df_clusters_GEP['cluster'].isin(range(10)), :]
# df_clusters_GEP.loc[df_clusters_GEP['cluster'] == 1, :]

# Save cluster map variables in a pickle.
#object_pi = {"clustermap": g, "cellscores_GEP_samples": data}
#file_pi = open(filename_clustermap, "wb") 
#pickle.dump(object_pi, file_pi)
#file_pi.close()

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import squareform

# Convert your compact pdist vector into a square distance matrix
distance_matrix = squareform(dist)

# Set the GEP string names as the index so .loc works flawlessly
df_clusters_GEP = df_clusters_GEP.set_index('sample')

# Extract your assigned cluster labels matching the matrix order exactly
cluster_labels = df_clusters_GEP.loc[data.index, 'cluster'].values

# Put the index back to a column just in case your downstream code expects it that way
df_clusters_GEP = df_clusters_GEP.reset_index()

# Compute the overall mean score for your paper
mean_sil = silhouette_score(distance_matrix, cluster_labels, metric='precomputed')
print(f"--- Silhouette Analysis for {age} ---")
print(f"Overall Average Silhouette Score: {mean_sil:.4f}\n")

# Compute individual scores for every single raw GEP program
sample_sil_values = silhouette_samples(distance_matrix, cluster_labels, metric='precomputed')

# Build a summary DataFrame for easy sorting and plotting
df_sil = pd.DataFrame({
    'GEP': data.index,
    'Cluster': cluster_labels,
    'Silhouette_Score': sample_sil_values
})

# Sort the programs primarily by Cluster, and secondarily by Silhouette Score descending
df_sil = df_sil.sort_values(by=['Cluster', 'Silhouette_Score'], ascending=[True, False]).reset_index(drop=True)

# Print average score per cluster to identify any under-clustered structural weaknesses
for i in sorted(df_sil['Cluster'].unique()):
    cluster_avg = df_sil[df_sil['Cluster'] == i]['Silhouette_Score'].mean()
    print(f"  * Mean Silhouette for Meta-Program Cluster {i}: {cluster_avg:.4f}")

### Find top genes for consensus GEPs that cluster across multiple patients and visualize them
Plot consensus GEPs on UMAP of each patient sample.


In [ ]:
age = "FiftyPlus"

gene_scores = GEP_zscore_samples
# gene_scores = GEP_tpm_samples  # column names do not match GEPs in df_clusters_GEP currently
# gene_scores.columns = GEP_zscore_samples.columns

# Identify mitochondrial genes, and drop them.
# genes_MT = gene_scores.index[gene_scores.index.str.contains("^MT-")].tolist()
# gene_scores.drop(genes_MT, axis=0, inplace=True)

# Group GEPs by assigned cluster from agglomerative clustering above, take average of each gene coefficient across GEPs, 
# sort genes by their average, keep top genes.
topgenes_GEP_consensus = []  # store consensus GEP programs[]
ngenes = 50
for name, group in df_clusters_GEP.groupby('cluster'):
    # Store GEPs, which were clustered together, in a dataframe.
    geps = group['sample']
    df_GEP = gene_scores[geps]
    
    # Get mean of each gene NMF score across all GEPs in the cluster, and sort genes by their mean.
    df_GEP.loc[:, 'mean'] = df_GEP.mean(axis = 1)
    df_GEP.sort_values(by='mean', ascending=False, inplace=True)
    topgenes_GEP_consensus.append(list(df_GEP.index[:ngenes]))

# Give names to consensus GEPs.
row_names = ['GEP_consensus_%s' % (i+1) for i in range(df_clusters_GEP['cluster'].nunique())]
topgenes_GEP_consensus = pd.DataFrame(topgenes_GEP_consensus, index=row_names).T
#topgenes_GEP_consensus.to_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus.csv", index=True)
topgenes_GEP_consensus.to_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus_SIX_META_PROGRAMS.csv", index=True)

# Place all top genes in a single list.
list_topgenes = []
for i in topgenes_GEP_consensus.columns:
    list_topgenes.extend(topgenes_GEP_consensus[i].tolist())

# Iterate over each sample and score each cell with the topgenes_GEP_consensus. 
for sampleid in sampleids_old:
    # Read in data, normalize, and log-transform the data.
    file_h5ad = "%s/data/individual_anndata/%s_%s_cNMF.h5ad" % (basedir, name_project, sampleid)
    adata = sc.read(file_h5ad)
    sc.pp.normalize_total(adata, target_sum=10**4) ## TPT normalization, rows sum to 10**4
    adata.raw = sc.pp.log1p(adata.copy(), copy=True)
    
    # Score cells with each GEP identified across all samples.
    for gep in topgenes_GEP_consensus.columns:
        gene_list = topgenes_GEP_consensus[gep].tolist()
        try:
            sc.tl.score_genes(adata, gene_list, ctrl_size=50, n_bins=25, score_name=gep, use_raw=True)
        except ValueError:  #  ValueError in score_genes() if no top genes expressed in this anndata object
            print(f"{gep} genes are not expressed in {sampleid}.")
            adata.obs[gep] = 0

    # UMAP and heatmap visualizations of genes.
    print(sampleid)
    gep_columns = adata.obs.columns[adata.obs.columns.str.contains("GEP_consensus")]
    sc.pl.umap(adata, color=gep_columns, use_raw=True, ncols=4, size = 12) 
    # sc.pl.umap(adata, color = ["leiden_labels", "SingleR_cluster_labels"], size = 12) 
    # sc.pl.heatmap(adata, var_names = list_topgenes, groupby="SingleR_cluster_labels")
    
# Save gene expression program variables in a pickle.
object_pi = {"topgenes_GEP_samples": topgenes_GEP_samples, "cellscores_GEP_samples": data, 
             "GEP_zscore_samples": GEP_zscore_samples, "topgenes_GEP_consensus": topgenes_GEP_consensus, "clustermap": g}
# file_pi = open(filename_pi, "wb") 
# pickle.dump(object_pi, file_pi)
# file_pi.close()

# Code to load data from pickled object.
# file_pi = open(filename_pi, "rb") 
# object_pi = pickle.load(file_pi)
# file_pi.close()
# GEP_zscore_samples = object_pi["GEP_zscore_samples"]

### Compare young and old consensus gene expression programs

In [ ]:
# Hierarchical/agglomerative clustering using scipy.cluster.hierarchy.linkage
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
import numpy as np

# Read in young and old consensus gene programs.
age = "UnderFifty"
topgenes_young_GEP_consensus = pd.read_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus.csv", index_col=0)

topgenes_young_GEP_consensus = topgenes_young_GEP_consensus.add_prefix('young_')
age = "FiftyPlus"
topgenes_old_GEP_consensus = pd.read_csv(f"{figuresdir}/topgenes_{age}_GEP_consensus.csv", index_col=0)

topgenes_old_GEP_consensus = topgenes_old_GEP_consensus.add_prefix('old_')


# Calculate Jaccard distance between pairs of young and old consensus gene programs and place in array.
def jaccard_distance(list1, list2):
    intersection = len(list(set(list1).intersection(list2)))
    union = (len(set(list1)) + len(set(list2))) - intersection
    return (1-float(intersection) / union)


for i in topgenes_young_GEP_consensus.columns:
    for j in topgenes_old_GEP_consensus.columns:
        d = jaccard_distance(topgenes_young_GEP_consensus[i], topgenes_old_GEP_consensus[j])
        print(f"{i} {j} {d}")

X = np.array(pd.concat([topgenes_young_GEP_consensus, topgenes_old_GEP_consensus], axis = 1))
dist = distance.pdist(X.T, jaccard_distance)
row_linkage = hierarchy.linkage(dist, method='ward', metric='euclidean')
col_linkage = hierarchy.linkage(dist.T, method='ward', metric='euclidean')

In [ ]:
# Look at overlap between colon aging genes and consensus gene programs.
file_gmt = "/home/EOCRC_atlas/docs/GTEx_colon_genes_aging.gmt"
dict_genes = {}
with open(file_gmt, "r") as file:
    for line in file:
        line = line.rstrip().split("\t")
        name = line[0]
        genes = line[2:]
        dict_genes[name] = genes

for i in topgenes_young_GEP_consensus.columns:
    d = len(set(topgenes_young_GEP_consensus[i]).intersection(dict_genes["young"]))
    print(f"{i} {d}")

for i in topgenes_old_GEP_consensus.columns:
    d = len(set(topgenes_old_GEP_consensus[i]).intersection(dict_genes["old"]))
    print(f"{i} {d}")

In [ ]:
cell_type = "Epithelial"
adata_proc = sc.read_h5ad(f'/home/EOCRC_atlas/data/yocrc_{cell_type}_annotation_noHarmony.h5ad')
adata_proc = adata_proc[(adata_proc.obs['Cohort']=='UnderFifty') & (adata_proc.obs['Sidedness']=='Left') & (adata_proc.obs['MSI_v2']=='MSS: STABLE') & (adata_proc.obs['Annotation_Tier2']!='Mixed - epithelial')]
# Look at genes from the consensus sets
# score each program and plot on adata_all 
# load in processed adata for epithelial cells 
for i in topgenes_young_GEP_consensus.columns:
    sc.tl.score_genes(adata_proc, topgenes_young_GEP_consensus[i].values, score_name = i, use_raw=True)

sc.pl.umap(adata_proc, color=topgenes_young_GEP_consensus.columns, size=2, cmap='inferno')

In [ ]:
sc.set_figure_params(figsize=(12, 6))
fig, ax = plt.subplots()
ax = sc.pl.violin(adata_proc, keys='young_GEP_consensus_1', groupby="FRID", ax=ax,
            use_raw=True, rotation=90, stripplot=False, inner="quartile")

sc.set_figure_params(figsize=(12, 6))
fig, ax = plt.subplots()
ax = sc.pl.violin(adata_proc, keys='young_GEP_consensus_2', groupby="FRID", ax=ax,
            use_raw=True, rotation=90, stripplot=False, inner="quartile")

sc.set_figure_params(figsize=(12, 6))
fig, ax = plt.subplots()
ax = sc.pl.violin(adata_proc, keys='young_GEP_consensus_3', groupby="FRID", ax=ax,
            use_raw=True, rotation=90, stripplot=False, inner="quartile")

sc.set_figure_params(figsize=(12, 6))
fig, ax = plt.subplots()
ax = sc.pl.violin(adata_proc, keys='young_GEP_consensus_4', groupby="FRID", ax=ax,
            use_raw=True, rotation=90, stripplot=False, inner="quartile")

sc.set_figure_params(figsize=(12, 6))
fig, ax = plt.subplots()
ax = sc.pl.violin(adata_proc, keys='young_GEP_consensus_5', groupby="FRID", ax=ax,
            use_raw=True, rotation=90, stripplot=False, inner="quartile")

In [ ]:
colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
for i in topgenes_young_GEP_consensus.columns:
    sc.set_figure_params(figsize=(4, 4))
    fig, ax = plt.subplots()
    ax = sc.pl.violin(adata_proc, keys=i, groupby="Annotation_Tier2", ax=ax,
                use_raw=True, rotation=90, stripplot=False, inner="quartile", palette = colors)
    fig.savefig(f'{figuresdir}/Violin_cNMF_underFifty_{date}_{i}_MSS_LEFT.pdf', dpi=600, bbox_inches='tight')
    plt.close(fig)

In [ ]:
topgenes_young_GEP_consensus

In [ ]:
cell_type = "Epithelial"
adata_proc = sc.read_h5ad(f'/home/EOCRC_atlas/data/yocrc_{cell_type}_annotation_noHarmony.h5ad')
adata_proc = adata_proc[(adata_proc.obs['Cohort']=='FiftyPlus') & (adata_proc.obs['Sidedness']=='Left') & (adata_proc.obs['MSI_v2']=='MSS: STABLE') & (adata_proc.obs['Annotation_Tier2']!='Mixed - epithelial')]

# Look at genes from the consensus sets
# score each program and plot on adata_all 
# load in processed adata for epithelial cells 
for i in topgenes_old_GEP_consensus.columns:
    sc.tl.score_genes(adata_proc, topgenes_old_GEP_consensus[i].values, score_name = i, use_raw=True)

sc.pl.umap(adata_proc, color=topgenes_old_GEP_consensus.columns, size=2, cmap='inferno')

In [ ]:
colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
for i in topgenes_old_GEP_consensus.columns:
    sc.set_figure_params(figsize=(4, 4))
    fig, ax = plt.subplots()
    ax = sc.pl.violin(adata_proc, keys=i, groupby="Annotation_Tier2", ax=ax,
                use_raw=True, rotation=90, stripplot=False, inner="quartile", palette = colors)
    fig.savefig(f'{figuresdir}/Violin_cNMF_FiftyPlus_{date}_{i}_MSS_LEFT.pdf', dpi=600, bbox_inches='tight')
    plt.close(fig)

In [ ]:
topgenes_old_GEP_consensus

In [ ]:
# confusion matrix showing jaccard distance 
jaccard_matrix = np.zeros((len(topgenes_young_GEP_consensus.columns), len(topgenes_old_GEP_consensus.columns)))

# Calculate the Jaccard distance and store it in the matrix
for i, young_gene in enumerate(topgenes_young_GEP_consensus.columns):
    for j, old_gene in enumerate(topgenes_old_GEP_consensus.columns):
        d = jaccard_distance(topgenes_young_GEP_consensus[young_gene], topgenes_old_GEP_consensus[old_gene])
        jaccard_matrix[i, j] = d
        print(f"{young_gene} {old_gene} {d}")

# Convert the matrix to a pandas DataFrame for better visualization
jaccard_df = pd.DataFrame(jaccard_matrix, 
                           index=topgenes_young_GEP_consensus.columns, 
                           columns=topgenes_old_GEP_consensus.columns)

jaccard_df

plt.figure(figsize=(5, 5))  # Optional: adjust the size of the heatmap
sns.heatmap(jaccard_df, annot=True, cmap='viridis', fmt=".2f", linewidths=0.5, cbar_kws={'label': 'Jaccard Distance'})
plt.grid(False)
plt.savefig(f'{figuresdir}/{date}_Jaccard_confusionMatrix.pdf', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# make stability summary plots 
stability_files = glob.glob(os.path.join(figuresdir, "**/*.k_selection_stats.df.npz"), recursive=True)

all_stability = []

for file in stability_files:
    current_frid = os.path.basename(file).split('.k_selection_stats')[0]

    # only pull MSS samples 
    if current_frid not in adata_all.obs['FRID'].unique():
        continue

    # laod the files 
    with np.load(file, allow_pickle=True) as data:
        df = pd.DataFrame(data['data'], columns=data['columns'])

    # set the data types  
    df['k'] = df['k'].astype(int)
    df['silhouette'] = df['silhouette'].astype(float)
    df['prediction_error'] = df['prediction_error'].astype(float)
    
    df = df.sort_values('k')
    df['FRID'] = current_frid
    all_stability.append(df)

all_stability_df = pd.concat(all_stability, ignore_index=True)
error_stats = all_stability_df.groupby('k')['prediction_error'].agg(['mean', 'sem']).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5)) 

# boxplot of k values 
sns.boxplot(data=all_stability_df, x='k', y='silhouette', color='gray', fill=False, width=0.6, ax=ax1, fliersize=0, zorder=2)
sns.stripplot(data=all_stability_df, x='k', y='silhouette', color='#2d3436', size=2.5, alpha=0.25, jitter=0.2, ax=ax1, zorder=1) # Added ax=ax1 parameter explicitly
ax1.set_ylabel("Stability")
ax1.set_xlabel("k")
ax1.set_ylim(0.60, 1.02) 

# mean error curve 
ax2.errorbar(x=range(len(error_stats['k'])), y=error_stats['mean'], yerr=error_stats['sem'],
             color='blue', linewidth=2.5, linestyle='-', marker='o', markersize=5, capsize=3, capthick=1.5, elinewidth=1.5, 
             label='Mean Error (±SEM)', zorder=3)
ax2.set_ylabel("Mean Prediction Error")
ax2.set_xlabel("k") # Added matching independent X-axis label

k_values = sorted(all_stability_df['k'].unique())
ax2.set_xticks(range(len(k_values)))
ax2.set_xticklabels(k_values)
ax2.tick_params(axis='y')

k_box_index = k_values.index(10)
ax1.axvline(x=k_box_index, color='#d63031', linestyle='--', linewidth=1.5, zorder=0)
ax2.axvline(x=k_box_index, color='#d63031', linestyle='--', linewidth=1.5, zorder=0)
ax1.grid(False)
ax2.grid(False)

plt.tight_layout() # Optimizes subplot spacing boundaries cleanly
plt.savefig(os.path.join(figuresdir, "summary_stability_error_MSS.pdf"), dpi=600)
plt.show()

In [ ]:
# Find usage values for chosen k and density 
usage_files = glob.glob(os.path.join(figuresdir, "**/*.usages.k_10.dt_0_2.consensus.txt"), recursive=True)

usage_data = {}
for file in usage_files:
    current_frid = os.path.basename(file).split('.')[0]
    
    # only pull MSS files 
    if current_frid not in adata_all.obs['FRID'].unique():
        continue
    
    df_usage = pd.read_csv(file, sep=r'\s+', index_col=0) 
    sorted_cols = sorted(df_usage.columns, key=lambda x: int(''.join(filter(str.isdigit, str(x)))))
    
    program_max_scores = []
    for col in sorted_cols:
        peak_score = np.percentile(df_usage[col], 95)
        program_max_scores.append(peak_score)
    usage_data[current_frid] = program_max_scores
        
master_heatmap_df = pd.DataFrame(usage_data)
master_heatmap_df.index = [f"GEP {i+1}" for i in range(master_heatmap_df.shape[0])]

mean_activity = master_heatmap_df.mean(axis=1)
master_heatmap_df = master_heatmap_df.reindex(mean_activity.sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(6.5, 3.2))

sns.heatmap(master_heatmap_df,cmap="YlOrRd", cbar_kws={'label': '95th Percentile Usages)'},
    xticklabels=False, yticklabels=True, ax=ax)

ax.set_xlabel(f"Left-sided MSS Tumors (n={master_heatmap_df.shape[1]})")
ax.set_ylabel("cNMF Programs")
ax.grid(False)
plt.tight_layout()

plt.savefig(os.path.join(figuresdir, "usage_scores_heatmap_MSS.pdf"), bbox_inches='tight', transparent=True, dpi=600)
plt.show()